In [ ]:
from typing import Dict,List,Any
class UserProfile:
    def __init__(self,goal:str,weight:int,experience_level:str):
        self.goal=goal
        self.weight=weight
        self.experience_level=experience_level
class QuestionRouter:
    def route(self,question:str) -> str:
        question_lower=question.lower()

        nutrition_keywards=["eat","diet","food","calorie","protein","fat loss","muscle gain"]
        workout_keywards=["train","workout","exercise","chest","legs","back","glutes"]

        for keyward in nutrition_keywards:
            if keyward in question_lower:
                return "nutrition"
            
        for keyward in workout_keywards:
            if keyward in question_lower:
                return "workout"
            
        return  "general"
class NutritionTool:
    def run(self,question:str,user_profile:UserProfile) -> str:
        question_lower=question.lower()

        if "fat loss" in question_lower:
            return  (
                f"For fat loss,focus on high protein,moderate carbs,"
                f"low fat,and calorie comtrol.Your current goal is{user_profile.goal}"
            )
        if "muscle gain" in question_lower:
            return (
                f"For muscle gain,eat enough protein and stay in a calorie surplus."
                f"Your weight is{user_profile.weight}"
            )
        return "Eat balanced meals with protein ,carbs,healthy fats,and vegetables."
class WorkoutTool:
    def run(self,question:str,user_profile:UserProfile) ->str:
        question_lower=question.lower()

        if "chest" in question_lower:
            return "For chest training,use bench press,incline dumbbell press,and cable fly."
        
        if "legs" in question_lower or "glutes" in question_lower:
            return "For legs and glutes,use squats,hip trusts,lunges,and Romanian deadlifts."
        
        return "Use progressive overload and choose training volume based on your level:{user_profile.experience_level}"
class SimpleRetriever:
    def __init__(self):
        self.documents=[
            "Fat loss requires a calories deficit and enough protein",
            "Muscle gain requires calories surplus,progressive overload and enough recovery.",
            "Chest training can include bench press,incline dumbbell press, and cable fly.",
            "Leg training can include squats,lunges,hip trusts,and Romanian deadlifts"
        ]
    def retrieve(self,question:str) ->List[str]:
        question_lower=question.lower().replace("?","")
        results=[]

        for doc in self.documents:
            doc_lower=doc.lower()
            if any(word in doc_lower for word in question_lower.split()):
                results.append(doc)
        return results[:2]
class PromptBuilder:
    def build(self,question:str,tool_result:str,retrieved_docs:List[str],user_profile:UserProfile) -> str:
        context="\n".join(retrieved_docs)
        prompt=f"""
    You are a helpful AI fitness assistant.

    UserProfile:
    Goal:{user_profile.goal}
    Weight:{user_profile.weight}
    Experience_level:{user_profile.experience_level}

    Question:
    {question}

    Tool Result:
    {tool_result}

    Retrieved Knowledge:
    {context}

    Give a clear and practical answer:
    """
        return prompt
class LocalLLM:
     def generate(self,prompt:str) -> str:
        return "This us where the LLM would generate the final answer based on the prompt"
class FitnessAIApp:
    def __init__(self):
        self.router=QuestionRouter()
        self.nutrition_tool=NutritionTool()
        self.workout_tool=WorkoutTool()
        self.retriever=SimpleRetriever()
        self.prompt_builder=PromptBuilder()
        self.llm=LocalLLM()
    def answer(self,question:str,user_profile:UserProfile) -> Dict[str,Any]:    
        route=self.router.route(question)
        if route == "nutrition":
            tool_result=self.nutrition_tool.run(question,user_profile)
        elif route == "workout":
            tool_result=self.workout_tool.run(question,user_profile)
        else:
            tool_result="This is a general fitness question"
        retrieved_docs=self.retriever.retrieve(question)
        prompt=self.prompt_builder.build(
            question=question,
            tool_result=tool_result,
            retrieved_docs=retrieved_docs,
            user_profile=user_profile
        )
        final_answer=self.llm.generate(prompt)
        
        return {
            "question":question,
            "route":route,
            "tool_result":tool_result,
            "retrieved_docs":retrieved_docs,
            "prompt":prompt,
            "final_answer":final_answer
        }
class AIAPIService:
    def __init__(self):
        self.app=FitnessAIApp()
    
    def validate_request(self,request_data:Dict[str,Any]) -> Dict[str,Any]:
        required_fields=[
            "question",
            "goal",
            "weight",
            "experience_level"
        ]
        missing_fields=[]
        for field in required_fields:
            if field not in request_data:
                missing_fields.append(field)
        if missing_fields:
            return {
                "valid":False,
                "error":f"Missing require field: {missing_fields}"
            }
        if not isinstance(request_data["question"],str):
            return {
                "valid":False,
                "error":"Question must be string"
            }
        if not isinstance(request_data["weight"],int):
            return {
                "valid":False,
                "erroe":"Weight must be integer"
            }
        return {
            "valid":True,
            "error":None
        }
    def handle_request(self,request_data:Dict[str,Any]) -> Dict[str,Any]:
        validation_result=self.validate_request(request_data)
        if validation_result["valid"] is False:
            return {
                "success":False,
                "status_code":400,
                "error":validation_result["error"],
                "data":None
            }
        user_profile=UserProfile(
            goal=request_data["goal"],
            weight=request_data["weight"],
            experience_level=request_data["experience_level"]
        )
        app_result=self.app.answer(
            question=request_data["question"],
            user_profile=user_profile
        )
        return {
            "success":True,
            "status_code":200,
            "error":None,
            "data":app_result
        }
api_service=AIAPIService()
request_data={
    "question":"What should I eat during fat loss?",
    "goal":"fat loss and muscle maintenance",
    "weight":90,
    "experience_level":"intermediate"
}
response=api_service.handle_request(request_data)
print("===== API RESPONSE =====")
print("success:",response["success"])
print("status_code:",response["status_code"])
print("error:",response["error"])

print("\n===== DATA =====")
print("question:",response["data"]["question"])
print("route:",response["data"]["route"])
print("tool_result:",response["data"]["tool_result"])
print("retrieved_docs:",response["data"]["retrieved_docs"])

print("\n===== PROMPT =====")
print(response["data"]["prompt"])

print("\n=====Final Answer =====")
print(response["data"]["final_answer"])

===== API RESPONSE =====
success: True
status_code: 200
error: None

===== DATA =====
question: What should I eat during fat loss?
route: nutrition
tool_result: ('For fat loss,focus on high protein,moderate carbs,', 'low fat,and calorie comtrol.Your current goal isfat loss and muscle maintenance')
retrieved_docs: ['Fat loss requires a calories deficit and enough protein', 'Muscle gain requires calories surplus,progressive overload and enough recovery.']

===== PROMPT =====

    You are a helpful AI fitness assistant.

    UserProfile:
    Goal:fat loss and muscle maintenance
    Weight:90
    Experience_level:intermediate

    Question:
    What should I eat during fat loss?

    Tool Result:
    ('For fat loss,focus on high protein,moderate carbs,', 'low fat,and calorie comtrol.Your current goal isfat loss and muscle maintenance')

    Retrieved Knowledge:
    Fat loss requires a calories deficit and enough protein
Muscle gain requires calories surplus,progressive overload and enough 

In [2]:
["eat","diet","food","calorie","protein","fat loss","muscle gain"]
["train","workout","exercise","chest","legs","back","glutes"]
"For fat loss,focus on high protein,moderate carbs,low fat,and calorie comtrol.Your current goal is{}"
"For muscle gain,eat enough protein and stay in a calorie surplus.Your weight is{}"
"Eat balanced meals with protein ,carbs,healthy fats,and vegetables."
"For chest training,use bench press,incline dumbbell press,and cable fly."
"For legs and glutes,use squats,hip trusts,lunges,and Romanian deadlifts."
"Use progressive overload and choose training volume based on your level:{}"
"Fat loss requires a calories deficit and enough protein"
"Muscle gain requires calories surplus,progressive overload and enough recovery."
"Chest training can include bench press,incline dumbbell press, and cable fly."
"Leg training can include squats,lunges,hip trusts,and Romanian deadlifts"
"This is a general fitness question"


'This is a general fitness question'